In [2]:
!unzip /content/RGB_P.zip

Archive:  /content/RGB_P.zip
   creating: RGB_P/
   creating: RGB_P/Patches/
   creating: RGB_P/Patches/Abnormal(Ulcer)/
  inflating: RGB_P/Patches/Abnormal(Ulcer)/1.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/10.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/100.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/101.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/102.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/103.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/104.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/105.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/106.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/107.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/108.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/109.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/11.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/110.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/111.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/112.jpg  
  inflating: RGB_P/Patches/Abnormal

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
# Config

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 1e-4
NUM_CLASSES = 2
SEED        = 42

BASE_DIR = "/content/RGB_P"

LABEL_MAP = {
    "Abnormal(Ulcer)"   : 1,   # DFU
    "Normal(Healthy skin)" : 0, # No DFU
    "Wound Images"      : 1,
    "Wound Images2"     : 1,
    "internetSet"       : 1,
    "samples"           : 1,
}

CLASS_NAMES = ["No DFU", "DFU"]

In [5]:
# Collect Image Paths + Labels

def collect_images(base_dir, label_map):
    image_paths, labels = [], []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}

    for root, dirs, files in os.walk(base_dir):
        folder_name = os.path.basename(root)
        if folder_name not in label_map:
            continue
        label = label_map[folder_name]
        for fname in files:
            if os.path.splitext(fname)[1].lower() in valid_exts:
                image_paths.append(os.path.join(root, fname))
                labels.append(label)

    return image_paths, labels

image_paths, labels = collect_images(BASE_DIR, LABEL_MAP)
label_counts = Counter(labels)
print(f"\nTotal images collected : {len(image_paths)}")
print(f"  No DFU (0) : {label_counts[0]}")
print(f"  DFU    (1) : {label_counts[1]}\n")



Total images collected : 1495
  No DFU (0) : 270
  DFU    (1) : 1225



In [6]:
# Transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [7]:
# Custom Dataset

class DFUDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [8]:
# Stratified Train / Val Split (80-20)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
indices = list(range(len(image_paths)))

for train_idx, val_idx in sss.split(indices, labels):
    train_paths  = [image_paths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_paths    = [image_paths[i] for i in val_idx]
    val_labels   = [labels[i] for i in val_idx]

print(f"Train : {len(train_paths)} images")
print(f"Val   : {len(val_paths)} images\n")

train_dataset = DFUDataset(train_paths, train_labels, transform=train_transforms)
val_dataset   = DFUDataset(val_paths,   val_labels,   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)


Train : 1196 images
Val   : 299 images



In [9]:
# Verify a Batch

sample_imgs, sample_lbls = next(iter(train_loader))
print(f"Sample batch — images: {sample_imgs.shape} | labels: {sample_lbls.shape}")
print(f"  Label distribution in batch: {Counter(sample_lbls.numpy())}\n")

Sample batch — images: torch.Size([32, 3, 224, 224]) | labels: torch.Size([32])
  Label distribution in batch: Counter({np.int64(1): 21, np.int64(0): 11})



In [10]:
# EfficientNet-B3 — Pretrained + Custom Head

weights = EfficientNet_B3_Weights.IMAGENET1K_V1
model   = models.efficientnet_b3(weights=weights)

# Freeze backbone — only train the classifier head for now
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features  # 1536 for B3
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(DEVICE)
print("EfficientNet-B3 loaded with custom classifier head.")
print(f"Classifier head:\n{model.classifier}\n")

# Confirm only head params are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}\n")


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 162MB/s]


EfficientNet-B3 loaded with custom classifier head.
Classifier head:
Sequential(
  (0): Dropout(p=0.4, inplace=True)
  (1): Linear(in_features=1536, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=256, out_features=2, bias=True)
)

Trainable params : 393,986 / 11,090,218



In [11]:
# Loss, Optimizer, Scheduler

class_counts  = [label_counts[0], label_counts[1]]
class_weights = torch.tensor(
    [1.0 / c for c in class_counts], dtype=torch.float
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [12]:
# Train Loop

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds        = outputs.argmax(dim=1)
        correct      += (preds == lbls).sum().item()
        total        += lbls.size(0)

    return running_loss / total, correct / total


def validate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs    = model(imgs)
            loss       = criterion(outputs, lbls)

            running_loss += loss.item() * imgs.size(0)
            preds        = outputs.argmax(dim=1)
            correct      += (preds == lbls).sum().item()
            total        += lbls.size(0)

    return running_loss / total, correct / total


def run_training(model, train_loader, val_loader,
                 optimizer, criterion, scheduler, num_epochs):
    history = {"train_loss": [], "val_loss": [],
               "train_acc":  [], "val_acc":  []}
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion)
        val_loss,   val_acc   = validate(
            model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch:02d}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_efficientnet_b3.pth")
            print(f"  Best model saved (Val Acc: {best_val_acc*100:.2f}%)")

    return history

print("Training loop defined. Pipeline verified end-to-end.")
print("   Call run_training() to start full training.\n")


Training loop defined. Pipeline verified end-to-end.
   Call run_training() to start full training.



In [13]:
# final Sanity Check

model.eval()
with torch.no_grad():
    test_out  = model(sample_imgs.to(DEVICE))
    test_pred = test_out.argmax(dim=1).cpu().numpy()
    test_prob = torch.softmax(test_out, dim=1).cpu().numpy()

print(f"Forward pass output shape : {test_out.shape}")  # should be [32, 2]
print(f"Sample predictions        : {[CLASS_NAMES[p] for p in test_pred[:6]]}")
print(f"Sample confidences        : {[round(test_prob[i][test_pred[i]] * 100, 2) for i in range(6)]}%")
print(f"\nModel summary:")
print(f"  Backbone   : EfficientNet-B3 (frozen)")
print(f"  Head input : {in_features} features")
print(f"  Output     : {NUM_CLASSES} classes {CLASS_NAMES}")
print(f"  Device     : {DEVICE}")
print(f"\nEfficientNet-B3 RGB pipeline verified end-to-end.")

Forward pass output shape : torch.Size([32, 2])
Sample predictions        : ['DFU', 'DFU', 'DFU', 'DFU', 'No DFU', 'DFU']
Sample confidences        : [np.float32(54.02), np.float32(50.97), np.float32(51.83), np.float32(54.51), np.float32(51.09), np.float32(55.02)]%

Model summary:
  Backbone   : EfficientNet-B3 (frozen)
  Head input : 1536 features
  Output     : 2 classes ['No DFU', 'DFU']
  Device     : cuda

EfficientNet-B3 RGB pipeline verified end-to-end.


In [14]:
# ── Unfreeze last 2 blocks of EfficientNet-B3 backbone ──────────────────────

for param in model.parameters():
    param.requires_grad = False

for block in list(model.features.children())[-2:]:
    for param in block.parameters():
        param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

backbone_params = [p for p in model.features.parameters() if p.requires_grad]
head_params     = list(model.classifier.parameters())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

# ── Differential LR optimizer ────────────────────────────────────────────────

optimizer = optim.Adam([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params,     "lr": 1e-4},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── Early stopping helper ─────────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.stop       = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

# ── Updated run_training with early stopping ─────────────────────────────────

def run_training(model, train_loader, val_loader,
                 optimizer, criterion, scheduler, num_epochs):
    history = {"train_loss": [], "val_loss": [],
               "train_acc":  [], "val_acc":  []}
    best_val_acc = 0.0
    es = EarlyStopping(patience=5)

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss,   val_acc   = validate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch:02d}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_efficientnet_b3_finetuned.pth")
            print(f"  ✓ Best model saved (Val Acc: {best_val_acc*100:.2f}%)")

        es.step(val_loss)
        if es.stop:
            print(f"  Early stopping triggered at epoch {epoch}")
            break

    return history

history = run_training(model, train_loader, val_loader,
                       optimizer, criterion, scheduler, NUM_EPOCHS)

Trainable: 4,271,100 / 11,090,218
Epoch [01/20] Train Loss: 0.5890 Acc: 88.46% | Val Loss: 0.4865 Acc: 96.99%
  ✓ Best model saved (Val Acc: 96.99%)
Epoch [02/20] Train Loss: 0.3844 Acc: 95.15% | Val Loss: 0.2946 Acc: 95.65%
Epoch [03/20] Train Loss: 0.2412 Acc: 95.40% | Val Loss: 0.1972 Acc: 95.32%
Epoch [04/20] Train Loss: 0.1662 Acc: 95.74% | Val Loss: 0.1490 Acc: 95.65%
Epoch [05/20] Train Loss: 0.1532 Acc: 95.65% | Val Loss: 0.1241 Acc: 96.99%
Epoch [06/20] Train Loss: 0.1197 Acc: 95.90% | Val Loss: 0.1087 Acc: 95.99%
Epoch [07/20] Train Loss: 0.1024 Acc: 96.91% | Val Loss: 0.0979 Acc: 96.32%
Epoch [08/20] Train Loss: 0.0857 Acc: 97.66% | Val Loss: 0.0872 Acc: 96.66%
Epoch [09/20] Train Loss: 0.0944 Acc: 96.91% | Val Loss: 0.0827 Acc: 96.66%
Epoch [10/20] Train Loss: 0.0982 Acc: 96.82% | Val Loss: 0.0758 Acc: 97.32%
  ✓ Best model saved (Val Acc: 97.32%)
Epoch [11/20] Train Loss: 0.0927 Acc: 96.74% | Val Loss: 0.0740 Acc: 96.66%
Epoch [12/20] Train Loss: 0.0897 Acc: 97.16% | Val L